In [3]:

import math
import pandas as pd
from collections import Counter

data = [
    ['Sunny', 'Hot', 'High', 'Weak', 'No'],
    ['Sunny', 'Hot', 'High', 'Strong', 'No'],
    ['Overcast', 'Hot', 'High', 'Weak', 'Yes'],
    ['Rain', 'Mild', 'High', 'Weak', 'Yes'],
    ['Rain', 'Cool', 'Normal', 'Weak', 'Yes'],
    ['Rain', 'Cool', 'Normal', 'Strong', 'No'],
    ['Overcast', 'Cool', 'Normal', 'Strong', 'Yes'],
    ['Sunny', 'Mild', 'High', 'Weak', 'No'],
    ['Sunny', 'Cool', 'Normal', 'Weak', 'Yes'],
    ['Rain', 'Mild', 'Normal', 'Weak', 'Yes'],
    ['Sunny', 'Mild', 'Normal', 'Strong', 'Yes'],
    ['Overcast', 'Mild', 'High', 'Strong', 'Yes'],
    ['Overcast', 'Hot', 'Normal', 'Weak', 'Yes'],
    ['Rain', 'Mild', 'High', 'Strong', 'No']
]

attributes = ['Outlook', 'Temperature', 'Humidity', 'Wind']
target = 'Play'

df = pd.DataFrame(data, columns=attributes + [target])


# Calculate entropy
def entropy(labels):
    counts = Counter(labels)
    total = len(labels)
    result = 0

    for count in counts.values():
        p = count / total
        result -= p * math.log2(p)

    return result


# Calculate information gain
def information_gain(df, attribute, target):
    total_entropy = entropy(df[target])
    weighted_entropy = 0

    for value in df[attribute].unique():
        subset = df[df[attribute] == value]
        weight = len(subset) / len(df)

        weighted_entropy += weight * entropy(subset[target])

    return total_entropy - weighted_entropy


# Build the complete ID3 decision tree
def id3(df, attributes, target):
    # If all examples have the same class
    if len(df[target].unique()) == 1:
        return df[target].iloc[0]

    # If no attributes remain, return majority class
    if len(attributes) == 0:
        return df[target].mode()[0]

    # Select the attribute with maximum information gain
    gains = {
        attribute: information_gain(df, attribute, target)
        for attribute in attributes
    }

    best_attribute = max(gains, key=gains.get)

    tree = {best_attribute: {}}
    remaining_attributes = [
        attribute for attribute in attributes
        if attribute != best_attribute
    ]

    # Recursively create branches for every attribute value
    for value in df[best_attribute].unique():
        subset = df[df[best_attribute] == value]

        if subset.empty:
            tree[best_attribute][value] = df[target].mode()[0]
        else:
            tree[best_attribute][value] = id3(
                subset,
                remaining_attributes,
                target
            )

    return tree


# Display the entire decision tree
def print_tree(tree, prefix="", is_last=True):
    if not isinstance(tree, dict):
        print(prefix + ("└── " if is_last else "├── ") + str(tree))
        return

    attribute = next(iter(tree))
    print(prefix + ("└── " if is_last else "├── ") + attribute)

    branches = list(tree[attribute].items())

    for i, (value, subtree) in enumerate(branches):
        last = (i == len(branches) - 1)

        print(prefix + ("    " if is_last else "│   ") +
              ("└── " if last else "├── ") + str(value))

        new_prefix = prefix + ("    " if is_last else "│   ")
        print_tree(subtree, new_prefix + ("    " if last else "│   "), True)


decision_tree = id3(df, attributes, target)

print("ID3 Decision Tree:")
print_tree(decision_tree)

# Execute ID3
decision_tree = id3(df, attributes, target)

print("Complete ID3 Decision Tree:")
print_tree(decision_tree)


# Display information gain for each attribute
print("\nInformation Gain:")
for attribute in attributes:
    print(attribute, "=", round(
        information_gain(df, attribute, target), 4
    ))


ID3 Decision Tree:
└── Outlook
    ├── Sunny
    │   └── Humidity
    │       ├── High
    │       │   └── No
    │       └── Normal
    │           └── Yes
    ├── Overcast
    │   └── Yes
    └── Rain
        └── Wind
            ├── Weak
            │   └── Yes
            └── Strong
                └── No
Complete ID3 Decision Tree:
└── Outlook
    ├── Sunny
    │   └── Humidity
    │       ├── High
    │       │   └── No
    │       └── Normal
    │           └── Yes
    ├── Overcast
    │   └── Yes
    └── Rain
        └── Wind
            ├── Weak
            │   └── Yes
            └── Strong
                └── No

Information Gain:
Outlook = 0.2467
Temperature = 0.0292
Humidity = 0.1518
Wind = 0.0481
